In [1]:
import pandas as pd

In [2]:
pub_info_path="../paper_fixed/paper_fixed_data.csv"
pub_funding_path="../../data/dropbox/LaborDivision/Data/deriveddata/funding/pub_full_funding.pkl"
cntry_region="../../data/dropbox/LaborDivision/Data/additionaldata/cntry_region.xlsx"
cntry_meta="../../data/dropbox/LaborDivision/Data/additionaldata/cntry_meta.csv"

In [3]:
filter_path='../../data/dropbox/LaborDivision/Data/deriveddata/funding/funding_regression_data.csv'

In [4]:
pub_info=pd.read_csv(pub_info_path)
pub_info=pub_info.iloc[: , 1:]
pub_info.head()

,id,year,dis,cluster_id,role,Code,gender,yfp,npub,F,L,M,ST
0,29768070,2008,51,20009795,F,JPN,F,2008,5,2,0,0,Advanced
1,31240446,2008,51,20009795,F,JPN,F,2008,5,2,0,0,Advanced
2,29768070,2008,51,29411586,M,JPN,M,2000,74,3,0,5,Advanced
3,31240446,2008,51,29411586,M,JPN,M,2000,74,3,0,5,Advanced
4,43269102,2015,51,29411586,M,JPN,M,2000,74,3,0,5,Advanced


In [5]:
pub_fund=pd.read_pickle(pub_funding_path)
pub_fund=pub_fund.rename(columns={'Code':'funder'})
pub_fund.head()

,id,funder
0,31429330,[CHN]
1,31430016,[VEN]
2,31430120,[BGD]
3,31430427,[EU]
4,31430891,[ITA]


In [6]:
pub_meta=pub_info.merge(pub_fund, on='id')

In [7]:
#get list of countries that are eu country
region=pd.read_excel(cntry_region)[['cntry','eulabel']]
eu=region[region.eulabel==1]
cntry_code=pd.read_csv(cntry_meta)[['cntry','Code']]
eu=eu.merge(cntry_code,on='cntry',how='left')
eulist=eu.Code.tolist()

In [8]:
def assign_fund(author_cntry, funder):
    label=0
    if author_cntry in funder:
        label=1
    elif 'EU' in funder and author_cntry in eulist:
        label=1
    return label
pub_meta['funded']=pub_meta.apply(lambda x: assign_fund(x.Code, x.funder),axis=1)

In [9]:
pub_meta

,id,year,dis,cluster_id,role,Code,gender,yfp,npub,F,L,M,ST,funder,funded
0,43269102,2015,51,29411586,M,JPN,M,2000,74,3,0,5,Advanced,[JPN],1
1,43269102,2015,51,492181,M,JPN,M,1983,287,0,11,43,Advanced,[JPN],1
2,43269102,2015,51,492181,M,JPN,M,1983,287,0,11,43,Advanced,[JPN],1
3,43269102,2015,51,7147306,M,JPN,M,1998,288,0,3,59,Advanced,[JPN],1
4,43269102,2015,51,7147306,M,JPN,M,1998,288,0,3,59,Advanced,[JPN],1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3405487,54102854,2015,8,2380869,L,PLW,M,2009,44,4,3,11,Lagging,"[USA, PLW]",1
3405488,54102854,2015,8,58821944,F,PLW,F,2011,8,4,0,2,Lagging,"[USA, PLW]",1
3405489,54102854,2015,8,58821944,F,PLW,F,2011,8,4,0,2,Lagging,"[USA, PLW]",1
3405490,54102854,2015,8,57162728,M,PLW,M,2015,2,0,0,1,Lagging,"[USA, PLW]",1


In [10]:
grouped=pub_meta.groupby('id')['funded'].nunique()

In [13]:
filtered_ids = grouped[grouped == 2].index.tolist()

In [14]:
filtered_df=pub_meta[pub_meta.id.isin(filtered_ids)]

In [15]:
filtered_df.drop(columns=['funder']).to_csv(filter_path)

In [16]:
len(filtered_df['id'].unique())/len(pub_info['id'].unique())

0.1545543723824497

In [70]:
len(filtered_df['id'].unique())

332618